# extraction et nettoyage

In [3]:
# ---------- Cellule 0 : configuration ----------
import os, sys, glob, warnings
warnings.filterwarnings("ignore")

# Dossier du Drive partagé (Yellow, Green, FHV)
DRIVE_DIR = r"G:\.shortcut-targets-by-id\1PwZMt2HEG0QPyAUvGDsPrwPrAUgJEAVp\Dataset_csv"
# Dossier sur ton PC (HVFHV et référentiel des zones)
LOCAL_DIR = r"C:\Users\DELL\Desktop\esprit\data_warehouse\projet\TLC"
# Dossier des résultats de l'objectif 6
OUT_DIR   = r"C:\Users\DELL\Desktop\esprit\data_warehouse\projet\ML6_segmentation"

CHEMINS = {
    "Yellow": os.path.join(DRIVE_DIR, "yellow_tripdata_2024-02_2024-03_2025-02_2025-03.csv"),
    "Green":  os.path.join(DRIVE_DIR, "green_tripdata_2024-02_2024-03_2025-02_2025-03.csv"),
    "FHV":    os.path.join(DRIVE_DIR, "fhv_tripdata_2024-02_2024-03_2025-02_2025-03.csv"),
    "HVFHV":  os.path.join(LOCAL_DIR, "fhvhv_tripdata_2024-02_2024-03_2025-02_2025-03.csv"),
}
ZONES_CSV = os.path.join(LOCAL_DIR, "taxi_zone_lookup.csv")

os.makedirs(OUT_DIR, exist_ok=True)

print("Résultats :", OUT_DIR, "\n")
for cat, chemin in CHEMINS.items():
    if os.path.exists(chemin):
        print(f"{cat:7s} OK      {os.path.getsize(chemin) / 1e6:>8,.0f} Mo   {chemin}")
    else:
        print(f"{cat:7s} ABSENT  {chemin}")
print(f"{'Zones':7s} {'OK' if os.path.exists(ZONES_CSV) else 'ABSENT'}   {ZONES_CSV}")

Résultats : C:\Users\DELL\Desktop\esprit\data_warehouse\projet\ML6_segmentation 

Yellow  OK         1,577 Mo   G:\.shortcut-targets-by-id\1PwZMt2HEG0QPyAUvGDsPrwPrAUgJEAVp\Dataset_csv\yellow_tripdata_2024-02_2024-03_2025-02_2025-03.csv
Green   OK            23 Mo   G:\.shortcut-targets-by-id\1PwZMt2HEG0QPyAUvGDsPrwPrAUgJEAVp\Dataset_csv\green_tripdata_2024-02_2024-03_2025-02_2025-03.csv
FHV     OK           497 Mo   G:\.shortcut-targets-by-id\1PwZMt2HEG0QPyAUvGDsPrwPrAUgJEAVp\Dataset_csv\fhv_tripdata_2024-02_2024-03_2025-02_2025-03.csv
HVFHV   OK           799 Mo   C:\Users\DELL\Desktop\esprit\data_warehouse\projet\TLC\fhvhv_tripdata_2024-02_2024-03_2025-02_2025-03.csv
Zones   OK   C:\Users\DELL\Desktop\esprit\data_warehouse\projet\TLC\taxi_zone_lookup.csv


#  colonnes utiles 
on choisit dans chaque fichier seulement les colonnes dont l’objectif 6 a besoin.

In [4]:

import time
import numpy as np
import pandas as pd

COLONNES = {
    "Yellow": {"pickup": "tpep_pickup_datetime", "dropoff": "tpep_dropoff_datetime",
               "pu": "PULocationID", "do": "DOLocationID",
               "distance": "trip_distance", "prix": "fare_amount",
               "pourboire": "tip_amount", "paiement": "payment_type",
               "cbd": "cbd_congestion_fee"},
    "Green":  {"pickup": "lpep_pickup_datetime", "dropoff": "lpep_dropoff_datetime",
               "pu": "PULocationID", "do": "DOLocationID",
               "distance": "trip_distance", "prix": "fare_amount",
               "pourboire": "tip_amount", "paiement": "payment_type",
               "cbd": "cbd_congestion_fee"},
    "FHV":    {"pickup": "pickup_datetime", "dropoff": "dropOff_datetime",
               "pu": "PUlocationID", "do": "DOlocationID"},
    "HVFHV":  {"pickup": "pickup_datetime", "dropoff": "dropoff_datetime",
               "pu": "PULocationID", "do": "DOLocationID",
               "distance": "trip_miles", "prix": "base_passenger_fare",
               "pourboire": "tips", "cbd": "cbd_congestion_fee",
               "request": "request_datetime", "partage": "shared_request_flag",
               "wav": "wav_request_flag", "remuneration": "driver_pay"},
}

def resoudre_colonnes(chemin, roles):
    entete = pd.read_csv(chemin, nrows=0).columns
    reel = {c.lower(): c for c in entete}
    manquantes = [c for c in roles.values() if c.lower() not in reel]
    if manquantes:
        raise ValueError(f"Colonnes absentes de {os.path.basename(chemin)} : {manquantes}")
    return {role: reel[c.lower()] for role, c in roles.items()}

for cat, chemin in CHEMINS.items():
    total = len(pd.read_csv(chemin, nrows=0).columns)
    noms = resoudre_colonnes(chemin, COLONNES[cat])
    print(f"{cat:7s} : {len(noms)} colonnes gardées sur {total} -> {list(noms.values())}")

Yellow  : 9 colonnes gardées sur 20 -> ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount', 'tip_amount', 'payment_type', 'cbd_congestion_fee']
Green   : 9 colonnes gardées sur 21 -> ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount', 'tip_amount', 'payment_type', 'cbd_congestion_fee']
FHV     : 4 colonnes gardées sur 7 -> ['pickup_datetime', 'dropOff_datetime', 'PUlocationID', 'DOlocationID']
HVFHV   : 12 colonnes gardées sur 25 -> ['pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'base_passenger_fare', 'tips', 'cbd_congestion_fee', 'request_datetime', 'shared_request_flag', 'wav_request_flag', 'driver_pay']


In [5]:
# ---------- Cellule 1 bis : extraction depuis le Drive -> row_data (colonnes utiles seulement) ----------
ROW_DIR = os.path.join(os.getcwd(), "row_data")       # dossier du notebook
os.makedirs(ROW_DIR, exist_ok=True)
A_EXTRAIRE = ["Yellow", "Green", "FHV"]                # les fichiers qui sont sur le Drive
BLOC_EXTRACTION = 500_000

def lire_par_blocs(chemin, colonnes, taille, essais_max=10):
    """Lit un CSV bloc par bloc ; si le Drive coupe la lecture, rouvre le fichier et reprend où il s'était arrêté."""
    deja_lues, essais = 0, 0
    while True:
        try:
            with open(chemin, "rb", buffering=4 * 1024 * 1024) as f:
                saut = (lambda i: 0 < i <= deja_lues) if deja_lues else None
                lecteur = pd.read_csv(f, usecols=colonnes, dtype=str, chunksize=taille, skiprows=saut)
                for bloc in lecteur:
                    deja_lues += len(bloc)
                    essais = 0
                    yield bloc
            return
        except OSError as e:
            essais += 1
            if essais > essais_max:
                raise
            print(f"\n   lecture coupée ({e}) – reprise à la ligne {deja_lues:,} (essai {essais}/{essais_max})")
            time.sleep(10)

for cat in A_EXTRAIRE:
    source = CHEMINS[cat]
    cible = os.path.join(ROW_DIR, f"{cat.lower()}_extrait.csv")
    marqueur = cible + ".ok"                           # créé seulement quand l'extraction est complète
    if os.path.exists(marqueur):
        print(f"{cat:7s} : déjà extrait -> {cible}")
    else:
        noms = resoudre_colonnes(source, COLONNES[cat])
        colonnes = list(noms.values())
        t0, n = time.time(), 0
        for i, bloc in enumerate(lire_par_blocs(source, colonnes, BLOC_EXTRACTION), start=1):
            bloc[colonnes].to_csv(cible, mode="w" if i == 1 else "a", header=(i == 1), index=False)
            n += len(bloc)
            print(f"\r{cat:7s} : {n:,} lignes copiées", end="")
        open(marqueur, "w").write(str(n))
        print(f"\n{cat:7s} : terminé en {time.time() - t0:,.0f} s – {os.path.getsize(cible)/1e9:.2f} Go")
    CHEMINS[cat] = cible                               # la suite du notebook lit la copie locale

print("\nFichiers utilisés pour la suite :")
for cat, chemin in CHEMINS.items():
    print(f"{cat:7s} : {chemin}")

Yellow  : 14,312,954 lignes copiées
Yellow  : terminé en 562 s – 1.15 Go
Green   : 209,194 lignes copiées
Green   : terminé en 7 s – 0.02 Go
FHV     : 6,407,159 lignes copiées
FHV     : terminé en 150 s – 0.38 Go

Fichiers utilisés pour la suite :
Yellow  : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\row_data\yellow_extrait.csv
Green   : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\row_data\green_extrait.csv
FHV     : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\row_data\fhv_extrait.csv
HVFHV   : C:\Users\DELL\Desktop\esprit\data_warehouse\projet\TLC\fhvhv_tripdata_2024-02_2024-03_2025-02_2025-03.csv


In [6]:
# ---------- Cellule 1 ter : extraction de HVFHV -> row_data (colonnes utiles seulement) ----------
ROW_DIR = os.path.join(os.getcwd(), "row_data")
os.makedirs(ROW_DIR, exist_ok=True)

source = CHEMINS["HVFHV"]
cible = os.path.join(ROW_DIR, "hvfhv_extrait.csv")
marqueur = cible + ".ok"

if os.path.exists(marqueur):
    print(f"HVFHV : déjà extrait -> {cible}")
else:
    noms = resoudre_colonnes(source, COLONNES["HVFHV"])
    colonnes = list(noms.values())
    print("Colonnes gardées :", colonnes)
    t0, n = time.time(), 0
    lecteur = pd.read_csv(source, usecols=colonnes, dtype=str, chunksize=500_000)
    for i, bloc in enumerate(lecteur, start=1):
        bloc[colonnes].to_csv(cible, mode="w" if i == 1 else "a", header=(i == 1), index=False)
        n += len(bloc)
        print(f"\rHVFHV : {n:,} lignes copiées", end="")
    open(marqueur, "w").write(str(n))
    print(f"\nHVFHV : terminé en {time.time() - t0:,.0f} s – {os.path.getsize(cible)/1e9:.2f} Go "
          f"(original : {os.path.getsize(source)/1e9:.2f} Go)")

CHEMINS["HVFHV"] = cible
print(f"HVFHV utilisé pour la suite : {CHEMINS['HVFHV']}")

Colonnes gardées : ['pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'base_passenger_fare', 'tips', 'cbd_congestion_fee', 'request_datetime', 'shared_request_flag', 'wav_request_flag', 'driver_pay']
HVFHV : 4,048,905 lignes copiées
HVFHV : terminé en 45 s – 0.46 Go (original : 0.80 Go)
HVFHV utilisé pour la suite : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\row_data\hvfhv_extrait.csv


In [7]:
# ---------- Étude 1 : aperçu des fichiers extraits + explication des variables ----------
from IPython.display import display, Markdown

ROW_DIR = os.path.join(os.getcwd(), "row_data")
FICHIERS = {
    "Yellow": os.path.join(ROW_DIR, "yellow_extrait.csv"),
    "Green":  os.path.join(ROW_DIR, "green_extrait.csv"),
    "FHV":    os.path.join(ROW_DIR, "fhv_extrait.csv"),
    "HVFHV":  os.path.join(ROW_DIR, "hvfhv_extrait.csv"),
}

NOTES = {
"Yellow": """
**Taxis jaunes (Yellow)** : taxis classiques, peuvent prendre un client partout dans NYC (hélés dans la rue), surtout à Manhattan et aux aéroports.

- **tpep_pickup_datetime** : date et heure où le compteur a été démarré (début de la course).
- **tpep_dropoff_datetime** : date et heure où le compteur a été arrêté (fin de la course) ; avec la date de début, on calcule la durée.
- **PULocationID** : numéro de la zone TLC où le client est monté (1 à 263 ; 264 et 265 = zone inconnue ou hors NYC).
- **DOLocationID** : numéro de la zone TLC où le client est descendu.
- **trip_distance** : distance parcourue en miles, mesurée par le taximètre.
- **fare_amount** : prix calculé par le compteur (temps + distance) en dollars, sans taxes, péages, suppléments ni pourboire.
- **tip_amount** : pourboire en dollars ; il n'est enregistré que pour les paiements par carte, les pourboires en espèces valent 0.
- **payment_type** : moyen de paiement codé (0 = Flex Fare, 1 = carte, 2 = espèces, 3 = sans frais, 4 = litige, 5 = inconnu, 6 = course annulée).
- **cbd_congestion_fee** : péage de congestion pour une course qui entre dans le centre de Manhattan (sous la 60e rue), appliqué depuis le 5 janvier 2025 ; vide ou 0 en 2024.
""",
"Green": """
**Taxis verts (Green / Boro taxis)** : taxis autorisés à prendre des clients dans la rue seulement hors du centre de Manhattan et hors aéroports (nord de Manhattan, Bronx, Brooklyn, Queens, Staten Island).

- **lpep_pickup_datetime** : date et heure de début de la course (même sens que pour Yellow, préfixe « lpep » au lieu de « tpep »).
- **lpep_dropoff_datetime** : date et heure de fin de la course.
- **PULocationID** : zone TLC de prise en charge.
- **DOLocationID** : zone TLC de dépose.
- **trip_distance** : distance en miles mesurée par le taximètre.
- **fare_amount** : prix du compteur en dollars, sans taxes, péages ni pourboire.
- **tip_amount** : pourboire en dollars, seulement pour les paiements par carte.
- **payment_type** : moyen de paiement codé (mêmes codes que Yellow : 1 = carte, 2 = espèces, ...).
- **cbd_congestion_fee** : péage de congestion du centre de Manhattan, à partir de janvier 2025.
""",
"FHV": """
**FHV classiques (For-Hire Vehicles)** : voitures avec chauffeur réservées à l'avance auprès d'une base (limousines, black cars, livery), hors grandes plateformes. Le fichier est très pauvre : pas de prix ni de distance.

- **pickup_datetime** : date et heure de prise en charge.
- **dropOff_datetime** : date et heure de dépose.
- **PUlocationID** : zone TLC de prise en charge ; souvent vide dans ce fichier, ces courses seront exclues (règle E3).
- **DOlocationID** : zone TLC de dépose ; souvent vide aussi.

Remarque : les noms de colonnes ont une casse différente (PUlocationID, dropOff_datetime) ; c'est pour ça qu'on les retrouve sans tenir compte des majuscules.
""",
"HVFHV": """
**HVFHV (High Volume For-Hire Vehicles)** : les grandes plateformes VTC (Uber, Lyft, ...) qui font plus de 10 000 courses par jour. C'est le fichier le plus riche et le plus volumineux.

- **pickup_datetime** : date et heure où le passager monte dans la voiture.
- **dropoff_datetime** : date et heure où le passager descend.
- **request_datetime** : date et heure où le passager a commandé la course dans l'application ; l'écart avec pickup_datetime donne le temps d'attente.
- **PULocationID** : zone TLC de prise en charge.
- **DOLocationID** : zone TLC de dépose.
- **trip_miles** : distance de la course en miles (équivalent de trip_distance).
- **base_passenger_fare** : prix de base payé par le passager en dollars, avant péages, taxes, frais et pourboire (équivalent de fare_amount).
- **tips** : pourboire donné au chauffeur en dollars.
- **cbd_congestion_fee** : péage de congestion du centre de Manhattan, à partir de janvier 2025.
- **shared_request_flag** : Y si le passager a demandé une course partagée (pool), N sinon.
- **wav_request_flag** : Y si le passager a demandé un véhicule accessible aux fauteuils roulants, N sinon.
- **driver_pay** : ce que le chauffeur a reçu pour la course en dollars, hors pourboire, péages et taxes ; comparé au prix, il donne la part qui revient au chauffeur.
""",
}

for cat, chemin in FICHIERS.items():
    apercu = pd.read_csv(chemin, nrows=5)
    display(Markdown(f"### {cat} — {len(apercu.columns)} colonnes — {os.path.getsize(chemin)/1e9:.2f} Go"))
    display(apercu)
    display(Markdown(NOTES[cat]))

### Yellow — 9 colonnes — 1.15 Go

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,payment_type,cbd_congestion_fee
0,2024-02-01 00:04:45.000000,2024-02-01 00:19:58.000000,68,236,4.39,20.5,1.28,1,NaN
1,2024-02-01 00:56:31.000000,2024-02-01 01:10:53.000000,48,243,7.71,31.0,9.00,1,NaN
2,2024-02-01 00:07:50.000000,2024-02-01 00:43:12.000000,132,261,28.69,70.0,0.00,2,NaN
3,2024-02-01 00:01:49.000000,2024-02-01 00:10:47.000000,161,163,1.10,9.3,2.85,1,NaN
4,2024-02-01 00:37:35.000000,2024-02-01 00:51:15.000000,246,79,2.60,15.6,0.00,2,NaN



**Taxis jaunes (Yellow)** : taxis classiques, peuvent prendre un client partout dans NYC (hélés dans la rue), surtout à Manhattan et aux aéroports.

- **tpep_pickup_datetime** : date et heure où le compteur a été démarré (début de la course).
- **tpep_dropoff_datetime** : date et heure où le compteur a été arrêté (fin de la course) ; avec la date de début, on calcule la durée.
- **PULocationID** : numéro de la zone TLC où le client est monté (1 à 263 ; 264 et 265 = zone inconnue ou hors NYC).
- **DOLocationID** : numéro de la zone TLC où le client est descendu.
- **trip_distance** : distance parcourue en miles, mesurée par le taximètre.
- **fare_amount** : prix calculé par le compteur (temps + distance) en dollars, sans taxes, péages, suppléments ni pourboire.
- **tip_amount** : pourboire en dollars ; il n'est enregistré que pour les paiements par carte, les pourboires en espèces valent 0.
- **payment_type** : moyen de paiement codé (0 = Flex Fare, 1 = carte, 2 = espèces, 3 = sans frais, 4 = litige, 5 = inconnu, 6 = course annulée).
- **cbd_congestion_fee** : péage de congestion pour une course qui entre dans le centre de Manhattan (sous la 60e rue), appliqué depuis le 5 janvier 2025 ; vide ou 0 en 2024.


### Green — 9 colonnes — 0.02 Go

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tip_amount,payment_type,cbd_congestion_fee
0,2024-02-01 00:15:53.000000,2024-02-01 00:24:20.000000,75,161,2.77,13.5,3.75,1,NaN
1,2024-01-31 22:59:22.000000,2024-01-31 23:27:14.000000,43,242,7.76,36.6,0.00,2,NaN
2,2024-02-01 00:30:29.000000,2024-02-01 00:35:32.000000,75,238,1.03,7.9,2.60,1,NaN
3,2024-01-31 23:56:42.000000,2024-02-01 00:06:53.000000,80,17,1.36,11.4,4.17,1,NaN
4,2024-02-01 00:31:14.000000,2024-02-01 00:31:16.000000,95,264,0.00,50.0,1.00,1,NaN



**Taxis verts (Green / Boro taxis)** : taxis autorisés à prendre des clients dans la rue seulement hors du centre de Manhattan et hors aéroports (nord de Manhattan, Bronx, Brooklyn, Queens, Staten Island).

- **lpep_pickup_datetime** : date et heure de début de la course (même sens que pour Yellow, préfixe « lpep » au lieu de « tpep »).
- **lpep_dropoff_datetime** : date et heure de fin de la course.
- **PULocationID** : zone TLC de prise en charge.
- **DOLocationID** : zone TLC de dépose.
- **trip_distance** : distance en miles mesurée par le taximètre.
- **fare_amount** : prix du compteur en dollars, sans taxes, péages ni pourboire.
- **tip_amount** : pourboire en dollars, seulement pour les paiements par carte.
- **payment_type** : moyen de paiement codé (mêmes codes que Yellow : 1 = carte, 2 = espèces, ...).
- **cbd_congestion_fee** : péage de congestion du centre de Manhattan, à partir de janvier 2025.


... (2 outputs left out)


**FHV classiques (For-Hire Vehicles)** : voitures avec chauffeur réservées à l'avance auprès d'une base (limousines, black cars, livery), hors grandes plateformes. Le fichier est très pauvre : pas de prix ni de distance.

- **pickup_datetime** : date et heure de prise en charge.
- **dropOff_datetime** : date et heure de dépose.
- **PUlocationID** : zone TLC de prise en charge ; souvent vide dans ce fichier, ces courses seront exclues (règle E3).
- **DOlocationID** : zone TLC de dépose ; souvent vide aussi.

Remarque : les noms de colonnes ont une casse différente (PUlocationID, dropOff_datetime) ; c'est pour ça qu'on les retrouve sans tenir compte des majuscules.


### HVFHV — 12 colonnes — 0.46 Go

,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,base_passenger_fare,tips,cbd_congestion_fee,request_datetime,shared_request_flag,wav_request_flag,driver_pay
0,2024-02-01 00:08:56.000000,2024-02-01 00:26:03.000000,179,226,2.520,15.18,0,NaN,2024-02-01 00:05:14.000000,N,N,12.97
1,2024-02-01 00:48:52.000000,2024-02-01 01:04:40.000000,211,233,2.901,28.36,0,NaN,2024-02-01 00:46:46.000000,N,N,21.13
2,2024-02-01 00:15:42.000000,2024-02-01 00:27:40.000000,74,167,3.400,17.00,0,NaN,2024-02-01 00:14:30.000000,N,N,11.22
3,2024-02-01 00:47:13.000000,2024-02-01 01:30:21.000000,164,189,10.490,52.65,0,NaN,2024-02-01 00:44:09.000000,N,N,38.11
4,2024-02-01 00:11:39.000000,2024-02-01 00:16:26.000000,14,14,1.500,8.89,1,NaN,2024-02-01 00:08:48.000000,N,N,5.39



**HVFHV (High Volume For-Hire Vehicles)** : les grandes plateformes VTC (Uber, Lyft, ...) qui font plus de 10 000 courses par jour. C'est le fichier le plus riche et le plus volumineux.

- **pickup_datetime** : date et heure où le passager monte dans la voiture.
- **dropoff_datetime** : date et heure où le passager descend.
- **request_datetime** : date et heure où le passager a commandé la course dans l'application ; l'écart avec pickup_datetime donne le temps d'attente.
- **PULocationID** : zone TLC de prise en charge.
- **DOLocationID** : zone TLC de dépose.
- **trip_miles** : distance de la course en miles (équivalent de trip_distance).
- **base_passenger_fare** : prix de base payé par le passager en dollars, avant péages, taxes, frais et pourboire (équivalent de fare_amount).
- **tips** : pourboire donné au chauffeur en dollars.
- **cbd_congestion_fee** : péage de congestion du centre de Manhattan, à partir de janvier 2025.
- **shared_request_flag** : Y si le passager a demandé une course partagée (pool), N sinon.
- **wav_request_flag** : Y si le passager a demandé un véhicule accessible aux fauteuils roulants, N sinon.
- **driver_pay** : ce que le chauffeur a reçu pour la course en dollars, hors pourboire, péages et taxes ; comparé au prix, il donne la part qui revient au chauffeur.


In [10]:
# ---------- Étude 2 : Yellow – chargement complet et renommage ----------
chemin = os.path.join(ROW_DIR, "yellow_extrait.csv")

RENOMMER_YELLOW = {
    "tpep_pickup_datetime":  "date_debut",
    "tpep_dropoff_datetime": "date_fin",
    "PULocationID":          "zone_depart",
    "DOLocationID":          "zone_arrivee",
    "trip_distance":         "distance",
    "fare_amount":           "prix",
    "tip_amount":            "pourboire",
    "payment_type":          "paiement",
    "cbd_congestion_fee":    "frais_cbd",
}
TYPES = {"PULocationID": "float32", "DOLocationID": "float32", "trip_distance": "float32",
         "fare_amount": "float32", "tip_amount": "float32", "payment_type": "float32",
         "cbd_congestion_fee": "float32"}
DATES = ["date_debut", "date_fin"]


display(yellow.head())

,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,paiement,frais_cbd
0,2024-02-01 00:04:45,2024-02-01 00:19:58,68.0,236.0,4.390000,20.5,1.28,1.0,NaN
1,2024-02-01 00:56:31,2024-02-01 01:10:53,48.0,243.0,7.710000,31.0,9.00,1.0,NaN
2,2024-02-01 00:07:50,2024-02-01 00:43:12,132.0,261.0,28.690001,70.0,0.00,2.0,NaN
3,2024-02-01 00:01:49,2024-02-01 00:10:47,161.0,163.0,1.100000,9.3,2.85,1.0,NaN
4,2024-02-01 00:37:35,2024-02-01 00:51:15,246.0,79.0,2.600000,15.6,0.00,2.0,NaN


In [ ]:

chemin = os.path.join(ROW_DIR, "green_extrait.csv")

RENOMMER_GREEN = {
    "lpep_pickup_datetime":  "date_debut",
    "lpep_dropoff_datetime": "date_fin",
    "PULocationID":          "zone_depart",
    "DOLocationID":          "zone_arrivee",
    "trip_distance":         "distance",
    "fare_amount":           "prix",
    "tip_amount":            "pourboire",
    "payment_type":          "paiement",
    "cbd_congestion_fee":    "frais_cbd",
}
TYPES = {"PULocationID": "float32", "DOLocationID": "float32", "trip_distance": "float32",
         "fare_amount": "float32", "tip_amount": "float32", "payment_type": "float32",
         "cbd_congestion_fee": "float32"}
DATES = ["date_debut", "date_fin"]

t0, morceaux = time.time(), []
for bloc in pd.read_csv(chemin, dtype=TYPES, chunksize=2_000_000):
    bloc = bloc.rename(columns=RENOMMER_GREEN)
    for c in DATES:
        bloc[c] = pd.to_datetime(bloc[c], format="ISO8601", errors="coerce")
    morceaux.append(bloc)
green = pd.concat(morceaux, ignore_index=True)

display(green.head())

,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,paiement,frais_cbd
0,2024-02-01 00:15:53,2024-02-01 00:24:20,75.0,161.0,2.77,13.500000,3.75,1.0,NaN
1,2024-01-31 22:59:22,2024-01-31 23:27:14,43.0,242.0,7.76,36.599998,0.00,2.0,NaN
2,2024-02-01 00:30:29,2024-02-01 00:35:32,75.0,238.0,1.03,7.900000,2.60,1.0,NaN
3,2024-01-31 23:56:42,2024-02-01 00:06:53,80.0,17.0,1.36,11.400000,4.17,1.0,NaN
4,2024-02-01 00:31:14,2024-02-01 00:31:16,95.0,264.0,0.00,50.000000,1.00,1.0,NaN


In [16]:

chemin = os.path.join(ROW_DIR, "fhv_extrait.csv")

RENOMMER_FHV = {
    "pickup_datetime":  "date_debut",
    "dropOff_datetime": "date_fin",
    "PUlocationID":     "zone_depart",
    "DOlocationID":     "zone_arrivee",
}
TYPES = {"PUlocationID": "float32", "DOlocationID": "float32"}
DATES = ["date_debut", "date_fin"]

t0, morceaux = time.time(), []
for bloc in pd.read_csv(chemin, dtype=TYPES, chunksize=2_000_000):
    bloc = bloc.rename(columns=RENOMMER_FHV)
    for c in DATES:
        bloc[c] = pd.to_datetime(bloc[c], format="ISO8601", errors="coerce")
    morceaux.append(bloc)
fhv = pd.concat(morceaux, ignore_index=True)

display(fhv.head())

,date_debut,date_fin,zone_depart,zone_arrivee
0,2024-02-01 00:00:00,2024-02-01 00:17:00,NaN,NaN
1,2024-02-01 00:30:00,2024-02-01 01:03:00,NaN,NaN
2,2024-02-01 00:00:00,2024-02-01 00:13:00,NaN,NaN
3,2024-02-01 00:00:00,2024-02-01 00:41:00,NaN,NaN
4,2024-02-01 00:46:26,2024-02-01 01:25:00,NaN,NaN


In [17]:

chemin = os.path.join(ROW_DIR, "hvfhv_extrait.csv")

RENOMMER_HVFHV = {
    "request_datetime":    "date_demande",
    "pickup_datetime":     "date_debut",
    "dropoff_datetime":    "date_fin",
    "PULocationID":        "zone_depart",
    "DOLocationID":        "zone_arrivee",
    "trip_miles":          "distance",
    "base_passenger_fare": "prix",
    "tips":                "pourboire",
    "cbd_congestion_fee":  "frais_cbd",
    "shared_request_flag": "partage",
    "wav_request_flag":    "acces_pmr",
    "driver_pay":          "paie_chauffeur",
}
TYPES = {"PULocationID": "float32", "DOLocationID": "float32", "trip_miles": "float32",
         "base_passenger_fare": "float32", "tips": "float32", "cbd_congestion_fee": "float32",
         "driver_pay": "float32", "shared_request_flag": "category", "wav_request_flag": "category"}
DATES = ["date_demande", "date_debut", "date_fin"]

t0, morceaux = time.time(), []
for i, bloc in enumerate(pd.read_csv(chemin, dtype=TYPES, chunksize=2_000_000), start=1):
    bloc = bloc[list(RENOMMER_HVFHV)].rename(columns=RENOMMER_HVFHV)
    for c in DATES:
        bloc[c] = pd.to_datetime(bloc[c], format="ISO8601", errors="coerce")
    morceaux.append(bloc)
    print(f"\rHVFHV : bloc {i} chargé", end="")
hvfhv = pd.concat(morceaux, ignore_index=True)
del morceaux

display(hvfhv.head())

HVFHV : bloc 3 chargé

,date_demande,date_debut,date_fin,zone_depart,zone_arrivee,distance,prix,pourboire,frais_cbd,partage,acces_pmr,paie_chauffeur
0,2024-02-01 00:05:14,2024-02-01 00:08:56,2024-02-01 00:26:03,179.0,226.0,2.520,15.180000,0.0,NaN,N,N,12.970000
1,2024-02-01 00:46:46,2024-02-01 00:48:52,2024-02-01 01:04:40,211.0,233.0,2.901,28.360001,0.0,NaN,N,N,21.129999
2,2024-02-01 00:14:30,2024-02-01 00:15:42,2024-02-01 00:27:40,74.0,167.0,3.400,17.000000,0.0,NaN,N,N,11.220000
3,2024-02-01 00:44:09,2024-02-01 00:47:13,2024-02-01 01:30:21,164.0,189.0,10.490,52.650002,0.0,NaN,N,N,38.110001
4,2024-02-01 00:08:48,2024-02-01 00:11:39,2024-02-01 00:16:26,14.0,14.0,1.500,8.890000,1.0,NaN,N,N,5.390000


sauvgarde les 4 fichiers

In [19]:

RENOM_DIR = os.path.join(os.getcwd(), "renamed_data")
os.makedirs(RENOM_DIR, exist_ok=True)

for cat, df in {"yellow": yellow, "green": green, "fhv": fhv, "hvfhv": hvfhv}.items():
    t0 = time.time()
    cible = os.path.join(RENOM_DIR, f"{cat}_renomme.csv")
    df.to_csv(cible, index=False, chunksize=1_000_000)
    print(f"{cat:6s} : {len(df):,} lignes, {df.shape[1]} colonnes -> {cible} "
          f"({os.path.getsize(cible)/1e9:.2f} Go, {time.time()-t0:,.0f} s)")

yellow : 14,312,954 lignes, 9 colonnes -> c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\renamed_data\yellow_renomme.csv (1.06 Go, 152 s)
green  : 209,194 lignes, 9 colonnes -> c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\renamed_data\green_renomme.csv (0.02 Go, 2 s)
fhv    : 6,407,159 lignes, 4 colonnes -> c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\renamed_data\fhv_renomme.csv (0.31 Go, 47 s)
hvfhv  : 4,048,905 lignes, 12 colonnes -> c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\renamed_data\hvfhv_renomme.csv (0.40 Go, 54 s)


# Le référentiel des zones
on charge la liste officielle des 265 zones de New York

In [3]:

zones = pd.read_csv(ZONES_CSV)
print(zones.shape)
zones.head()

(265, 4)


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


# Les règles du nettoyage
fixe les seuils

In [4]:
PERIODES    = [("2024-02-01", "2024-04-01"), ("2025-02-01", "2025-04-01")]
ZONE_MIN, ZONE_MAX = 1, 263
AEROPORTS   = [1, 132, 138]          # EWR, JFK, LaGuardia
DUREE_MIN, DUREE_MAX = 1, 180        # minutes
DIST_MAX    = 100                    # miles
PRIX_MAX    = 500                    # dollars
ATTENTE_MAX = 3600                   # secondes
CHUNK       = 1_000_000              # lignes lues à la fois

TRANCHES        = ["nuit", "matin", "journee", "soir", "soiree"]   # 0-6, 6-10, 10-16, 16-20, 20-24
BORNES_TRANCHES = [0, 6, 10, 16, 20, 24]
CLES_RAPPORT = ["lues", "D1_doublons", "E1_date_illisible", "E2_hors_periode", "E3_zone_invalide",
                "gardees", "M1_duree_invalide", "M2_distance_invalide", "M3_prix_invalide",
                "M4_attente_invalide"]

# Les fonctions de nettoyage
on écrit les deux fonctions qui nettoient un bloc de courses, puis le résument par zone.

In [5]:
# ---------- Cellule 4 : fonctions de nettoyage et de résumé ----------
def lire_dates(serie):
    return pd.to_datetime(serie, format="ISO8601", errors="coerce")


def retirer_doublons(bloc, deja_vus, rapport):
    """Retire les courses identiques (dans le bloc ou déjà vues dans un bloc précédent)."""
    empreintes = pd.util.hash_pandas_object(bloc, index=False).to_numpy()
    doublon = pd.Series(empreintes).duplicated().to_numpy() | np.isin(empreintes, deja_vus)
    rapport["D1_doublons"] += int(doublon.sum())
    deja_vus = np.union1d(deja_vus, empreintes[~doublon])
    return bloc.loc[~doublon], deja_vus


def nettoyer_et_resumer(bloc, rapport):
    """Applique les règles E1-E3 et M1-M4, puis résume le bloc par zone de départ."""
    pickup = lire_dates(bloc["pickup"])
    ok_date = pickup.notna()
    rapport["E1_date_illisible"] += int((~ok_date).sum())

    dans_periode = pd.Series(False, index=bloc.index)
    for debut, fin in PERIODES:
        dans_periode |= (pickup >= debut) & (pickup < fin)
    rapport["E2_hors_periode"] += int((ok_date & ~dans_periode).sum())

    pu = pd.to_numeric(bloc["pu"], errors="coerce")
    ok_zone = pu.between(ZONE_MIN, ZONE_MAX)
    rapport["E3_zone_invalide"] += int((dans_periode & ~ok_zone).sum())

    garde = dans_periode & ok_zone
    b, pickup, pu = bloc.loc[garde], pickup[garde], pu[garde].astype(int)
    rapport["gardees"] += len(b)
    if len(b) == 0:
        return None

    out = pd.DataFrame({"pu": pu.values})
    out["n"] = 1
    out["n_2024"] = (pickup.dt.year == 2024).astype(int).values
    out["n_2025"] = (pickup.dt.year == 2025).astype(int).values
    out["n_weekend"] = (pickup.dt.dayofweek >= 5).astype(int).values
    tranche = pd.cut(pickup.dt.hour, bins=BORNES_TRANCHES, labels=TRANCHES, right=False)
    for t in TRANCHES:
        out[f"n_{t}"] = (tranche == t).astype(int).values
    do = pd.to_numeric(b["do"], errors="coerce")
    out["n_aeroport"] = do.isin(AEROPORTS).astype(int).values

    # M1 – durée
    dropoff = lire_dates(b["dropoff"])
    duree = (dropoff - pickup).dt.total_seconds() / 60
    ok_duree = duree.between(DUREE_MIN, DUREE_MAX)
    rapport["M1_duree_invalide"] += int((~ok_duree).sum())
    out["n_duree"] = ok_duree.astype(int).values
    out["s_duree"] = duree.where(ok_duree, 0).values

    # M2 / M3 – distance et prix (taxis et HVFHV)
    if "distance" in b.columns:
        dist = pd.to_numeric(b["distance"], errors="coerce")
        prix = pd.to_numeric(b["prix"], errors="coerce")
        ok_dist = (dist > 0) & (dist <= DIST_MAX)
        ok_prix = (prix > 0) & (prix <= PRIX_MAX)
        rapport["M2_distance_invalide"] += int((~ok_dist).sum())
        rapport["M3_prix_invalide"] += int((~ok_prix).sum())
        out["n_dist"] = ok_dist.astype(int).values
        out["s_dist"] = dist.where(ok_dist, 0).values
        out["n_prix"] = ok_prix.astype(int).values
        out["s_prix"] = prix.where(ok_prix, 0).values
        ok_pm = ok_dist & ok_prix
        out["s_prix_pm"] = prix.where(ok_pm, 0).values
        out["s_dist_pm"] = dist.where(ok_pm, 0).values

        # pourboire (taux = pourboires / prix, sur les courses au prix valide)
        pourboire = pd.to_numeric(b["pourboire"], errors="coerce")
        ok_pb = ok_prix & pourboire.between(0, PRIX_MAX)
        out["s_pourboire"] = pourboire.where(ok_pb, 0).values
        out["s_prix_pb"] = prix.where(ok_pb, 0).values

        # taxe CBD (part des courses 2025 qui l'ont payée)
        cbd = pd.to_numeric(b["cbd"], errors="coerce")
        an_2025 = pickup.dt.year == 2025
        out["n_base_cbd"] = an_2025.astype(int).values
        out["n_cbd_taxee"] = (an_2025 & (cbd > 0)).astype(int).values

    # paiement (taxis) : part payée en espèces (code 2)
    if "paiement" in b.columns:
        paiement = pd.to_numeric(b["paiement"], errors="coerce")
        out["n_paiement"] = paiement.notna().astype(int).values
        out["n_especes"] = (paiement == 2).astype(int).values

    # M4 – attente, partage, accessibilité, rémunération (HVFHV)
    if "request" in b.columns:
        demande = lire_dates(b["request"])
        attente = (pickup - demande).dt.total_seconds()
        ok_att = attente.between(0, ATTENTE_MAX)
        rapport["M4_attente_invalide"] += int((~ok_att).sum())
        out["n_attente"] = ok_att.astype(int).values
        out["s_attente"] = attente.where(ok_att, 0).values
        out["n_hvfhv"] = 1
        out["n_partage"] = (b["partage"].astype(str).str.strip().str.upper() == "Y").astype(int).values
        out["n_wav"] = (b["wav"].astype(str).str.strip().str.upper() == "Y").astype(int).values
        remu = pd.to_numeric(b["remuneration"], errors="coerce")
        ok_remu = ok_prix & remu.between(0, PRIX_MAX)
        out["s_remuneration"] = remu.where(ok_remu, 0).values
        out["s_prix_remu"] = prix.where(ok_remu, 0).values

    return out.groupby("pu").sum()

retirer_doublons calcule une « empreinte » de chaque course (un nombre unique calculé à partir de toutes ses colonnes). Deux courses identiques ont la même empreinte. On retire celles déjà vues dans le bloc ou dans un bloc précédent. Avant : des courses en double possibles ; après : chaque course est unique.

nettoyer_et_resumer retire les courses à la date illisible, hors période ou sans zone valide. Pour chaque course gardée, elle note dans des colonnes 0/1 ou des valeurs : l’année, le week-end, la tranche horaire, l’aéroport, la durée, la distance, le prix, le pourboire, la taxe CBD, le paiement en espèces et, pour HVFHV, l’attente, le partage, la demande de véhicule accessible et la rémunération. Puis elle additionne tout par zone. Avant : 1 million de courses ; après : environ 260 lignes, une par zone, avec des compteurs et des sommes.

# lecture par blocs, nettoyage et agrégation

In [1]:
# ---------- Cellule 1 bis : extraction depuis le Drive -> row_data (colonnes utiles seulement) ----------
ROW_DIR = os.path.join(os.getcwd(), "row_data")       # dossier du notebook
os.makedirs(ROW_DIR, exist_ok=True)
A_EXTRAIRE = ["Yellow", "Green", "FHV"]                # les fichiers qui sont sur le Drive
BLOC_EXTRACTION = 500_000

def lire_par_blocs(chemin, colonnes, taille, essais_max=10):
    """Lit un CSV bloc par bloc ; si le Drive coupe la lecture, rouvre le fichier et reprend où il s'était arrêté."""
    deja_lues, essais = 0, 0
    while True:
        try:
            with open(chemin, "rb", buffering=4 * 1024 * 1024) as f:
                saut = (lambda i: 0 < i <= deja_lues) if deja_lues else None
                lecteur = pd.read_csv(f, usecols=colonnes, dtype=str, chunksize=taille, skiprows=saut)
                for bloc in lecteur:
                    deja_lues += len(bloc)
                    essais = 0
                    yield bloc
            return
        except OSError as e:
            essais += 1
            if essais > essais_max:
                raise
            print(f"\n   lecture coupée ({e}) – reprise à la ligne {deja_lues:,} (essai {essais}/{essais_max})")
            time.sleep(10)

for cat in A_EXTRAIRE:
    source = CHEMINS[cat]
    cible = os.path.join(ROW_DIR, f"{cat.lower()}_extrait.csv")
    marqueur = cible + ".ok"                           # créé seulement quand l'extraction est complète
    if os.path.exists(marqueur):
        print(f"{cat:7s} : déjà extrait -> {cible}")
    else:
        noms = resoudre_colonnes(source, COLONNES[cat])
        colonnes = list(noms.values())
        t0, n = time.time(), 0
        for i, bloc in enumerate(lire_par_blocs(source, colonnes, BLOC_EXTRACTION), start=1):
            bloc[colonnes].to_csv(cible, mode="w" if i == 1 else "a", header=(i == 1), index=False)
            n += len(bloc)
            print(f"\r{cat:7s} : {n:,} lignes copiées", end="")
        open(marqueur, "w").write(str(n))
        print(f"\n{cat:7s} : terminé en {time.time() - t0:,.0f} s – {os.path.getsize(cible)/1e9:.2f} Go")
    CHEMINS[cat] = cible                               # la suite du notebook lit la copie locale

print("\nFichiers utilisés pour la suite :")
for cat, chemin in CHEMINS.items():
    print(f"{cat:7s} : {chemin}")

NameError: name 'os' is not defined